In [1]:
pip install imbalanced-learn

In [2]:
pip install flask_cors

In [4]:
import numpy as np
import pandas as pd

# ── Check xlsx first ───────────────────────────────────────────────────────────
df = pd.read_excel("/content/muril1.xlsx", engine="openpyxl")
print(f"xlsx rows: {len(df)}")
print(f"xlsx columns: {list(df.columns)}")
print(f"\nLanguage counts:\n{df['language'].value_counts()}")

# ── Check raw npy file size ────────────────────────────────────────────────────
import os
size = os.path.getsize("/content/muril1.npy")
print(f"\nnpy file size: {size:,} bytes ({size/1024/1024:.1f} MB)")

# ── Try loading with different methods ────────────────────────────────────────
try:
    X = np.load("/content/muril1.npy", allow_pickle=True,
                fix_imports=True)
    print(f"npy shape: {X.shape}")
except Exception as e:
    print(f"Error: {e}")

# ── Check what dimensions actually fit ────────────────────────────────────────
total_values = 27787232
print(f"\nTotal values in npy: {total_values:,}")
print(f"If rows=76949: dim = {total_values/76949:.1f}")
print(f"If dim=768:    rows = {total_values/768:.1f}")
print(f"If dim=384:    rows = {total_values/384:.1f}")
print(f"If dim=512:    rows = {total_values/512:.1f}")

xlsx rows: 76949
xlsx columns: ['text', 'language', 'label']

Language counts:
language
Hindi       20593
Telugu      20001
Marathi     18696
Gujarati    17659
Name: count, dtype: int64

npy file size: 236,387,456 bytes (225.4 MB)
npy shape: (76949, 768)

Total values in npy: 27,787,232
If rows=76949: dim = 361.1
If dim=768:    rows = 36181.3
If dim=384:    rows = 72362.6
If dim=512:    rows = 54271.9


In [5]:
import numpy as np

# ── Load with correct shape ────────────────────────────────────────────────────
raw = np.fromfile("/content/muril1.npy", dtype=np.float32)
print(f"Total floats: {len(raw):,}")
print(f"76949 × 361 = {76949 * 361:,}")
print(f"76949 × 362 = {76949 * 362:,}")

# Try reshaping
for dim in [361, 362, 356, 358, 360]:
    if len(raw) % dim == 0:
        rows = len(raw) // dim
        print(f"dim={dim} → rows={rows}")

Total floats: 59,096,864
76949 × 361 = 27,778,589
76949 × 362 = 27,855,538


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

# ── Load xlsx ──────────────────────────────────────────────────────────────────
df = pd.read_excel("/content/muril1.xlsx", engine="openpyxl")
df.columns = df.columns.str.strip().str.lower()
df = df.dropna(subset=["label"]).reset_index(drop=True)
texts = df["text"].fillna("").tolist()
print(f"Generating embeddings for {len(texts)} texts…")

# ── Load MuRIL and generate embeddings ────────────────────────────────────────
model = SentenceTransformer("google/muril-base-cased")
X     = model.encode(texts, batch_size=64,
                     show_progress_bar=True,
                     convert_to_numpy=True)
print(f"Embeddings shape: {X.shape}")

# ── Save new npy ───────────────────────────────────────────────────────────────
np.save("/content/muril1.npy", X)
print(f"Saved → /content/muril1.npy")
print(f"Shape: {X.shape}  — rows={X.shape[0]}  dim={X.shape[1]}")

Generating embeddings for 76949 texts…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Batches:   0%|          | 0/1203 [00:00<?, ?it/s]

Embeddings shape: (76949, 768)
Saved → /content/muril1.npy
Shape: (76949, 768)  — rows=76949  dim=768


In [ ]:
"""
train.py — Splits muril1.npy / muril1.xlsx by language, trains
classifiers per language with proper train/test split,
feature engineering, hyperparameter tuning, ROC-AUC,
learning curves, and confusion matrix.

YOUR FILES (same folder as this script):
  muril1.npy    ← all 4 language embeddings
  muril1.xlsx   ← columns: text, language, label

OUTPUT:
  models/
    hindi1_model.pkl
    marathi1_model.pkl
    telugu1_model.pkl
    gujarati1_model.pkl
    training_report.json
    best_config.json
    lang_config.json
    <lang>_learning_curve.json
"""

import os, json, time, warnings, pickle, logging, re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    train_test_split, GridSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────────
HERE      = Path("/content")        # ← change this line only
MODEL_DIR = HERE / "models"
MODEL_DIR.mkdir(exist_ok=True)
# ── Language name in xlsx → model save name ───────────────────────────────────
LANGUAGE_MAP = {
    "hindi":    "hindi1",
    "marathi":  "marathi1",
    "telugu":   "telugu1",
    "gujarati": "gujarati1",
}

# ── Fake/real signal words per language ───────────────────────────────────────
FAKE_SIGNALS = {
    "hindi":    ["वायरल", "खुलासा", "चौंकाने", "शेयर", "सच्चाई",
                 "breaking", "viral", "exclusive", "shocking",
                 "अलर्ट", "सावधान", "दावा", "झूठ"],
    "marathi":  ["व्हायरल", "खुलासा", "धक्कादायक", "शेअर",
                 "सत्य", "breaking", "viral", "दावा", "खोटे"],
    "telugu":   ["వైరల్", "షాకింగ్", "వెల్లడి", "షేర్",
                 "నిజం", "breaking", "viral", "దావా"],
    "gujarati": ["વાયરલ", "ખુલાસો", "ચોંકાવનારું", "શેર",
                 "સત્ય", "breaking", "viral", "દાવો", "ખોટા"],
}

REAL_SIGNALS = {
    "hindi":    ["पीटीआई", "एएनआई", "संसद", "सरकार", "अधिकारी",
                 "बयान", "रिपोर्ट", "PTI", "ANI", "प्रेस"],
    "marathi":  ["पीटीआई", "एएनआई", "संसद", "सरकार", "अधिकारी",
                 "निवेदन", "अहवाल", "PTI", "ANI"],
    "telugu":   ["పిటిఐ", "ప్రభుత్వం", "పార్లమెంట్", "అధికారి",
                 "నివేదిక", "PTI", "ANI", "ప్రకటన"],
    "gujarati": ["પીટીઆઈ", "સરકાર", "સંસદ", "અધિકારી",
                 "નિવેદન", "અહેવાલ", "PTI", "ANI"],
}

# ── 10 classifiers with calibration ──────────────────────────────────────────
CLASSIFIERS = {
    "LogisticRegression": LogisticRegression(
                              max_iter=1000, C=1.0),
    "LinearSVC":          CalibratedClassifierCV(
                              LinearSVC(max_iter=2000, C=1.0), cv=3),
    "SVC_RBF":            CalibratedClassifierCV(
                              SVC(kernel="rbf", C=1.0), cv=3),
    "SGD":                CalibratedClassifierCV(
                              SGDClassifier(max_iter=1000,
                                            loss="modified_huber"),
                              cv=3),
    "RandomForest":       RandomForestClassifier(
                              n_estimators=200, n_jobs=-1,
                              random_state=42),
    "ExtraTrees":         ExtraTreesClassifier(
                              n_estimators=200, n_jobs=-1,
                              random_state=42),
    "GradientBoosting":   GradientBoostingClassifier(
                              n_estimators=100),
    "AdaBoost":           AdaBoostClassifier(
                              n_estimators=100),
    "GaussianNB":         GaussianNB(),
    "MLP":                MLPClassifier(
                              hidden_layer_sizes=(256, 128),
                              max_iter=500),
}

# ── Hyperparameter grids ───────────────────────────────────────────────────────
PARAM_GRIDS = {
    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1.0, 10.0]
    },
    "RandomForest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth":    [None, 10, 20],
    },
    "MLP": {
        "clf__hidden_layer_sizes": [(128,), (256, 128), (512, 256)],
        "clf__alpha":              [0.0001, 0.001],
    },
    "ExtraTrees": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth":    [None, 10, 20],
    },
}

CV_FOLDS     = 5
RANDOM_STATE = 42
TEST_SIZE    = 0.2


# ── Handcrafted feature engineering ───────────────────────────────────────────
def extract_handcrafted_features(texts, lang):
    """
    Hand-engineered linguistic features designed to capture
    fake vs real news patterns. Combined with MuRIL embeddings
    to create a richer feature set.
    """
    fake_words = FAKE_SIGNALS.get(lang, FAKE_SIGNALS["hindi"])
    real_words = REAL_SIGNALS.get(lang, REAL_SIGNALS["hindi"])
    features   = []

    for text in texts:
        f    = {}
        text = str(text)

        # ── Lexical features ──────────────────────────────────────────
        words = text.split()
        f["word_count"]        = len(words)
        f["char_count"]        = len(text)
        f["avg_word_length"]   = (sum(len(w) for w in words)
                                  / max(len(words), 1))
        f["unique_word_ratio"] = (len(set(words))
                                  / max(len(words), 1))

        # ── Punctuation features ──────────────────────────────────────
        f["exclamation_count"] = text.count("!")
        f["question_count"]    = text.count("?")
        f["ellipsis_count"]    = text.count("...")
        f["caps_ratio"]        = (sum(1 for c in text if c.isupper())
                                  / max(len(text), 1))

        # ── Language-specific signal words ────────────────────────────
        text_lower = text.lower()
        f["fake_signal_count"] = sum(
            1 for w in fake_words if w.lower() in text_lower)
        f["real_signal_count"] = sum(
            1 for w in real_words if w.lower() in text_lower)
        f["signal_ratio"]      = (
            f["fake_signal_count"] /
            max(f["fake_signal_count"] + f["real_signal_count"], 1)
        )

        # ── Structural features ───────────────────────────────────────
        sentences = re.split(r"[।.!?]", text)
        sentences = [s.strip() for s in sentences if s.strip()]
        f["sentence_count"]      = len(sentences)
        f["avg_sentence_length"] = (
            sum(len(s.split()) for s in sentences)
            / max(len(sentences), 1)
        )

        # ── URL and number features ───────────────────────────────────
        f["url_count"]    = len(re.findall(r"http\S+|www\.\S+", text))
        f["number_count"] = len(re.findall(r"\d+", text))

        # ── Repetition feature ────────────────────────────────────────
        # Fake news often repeats keywords for emphasis
        word_freq  = {}
        for w in words:
            word_freq[w] = word_freq.get(w, 0) + 1
        f["max_word_repetition"] = (max(word_freq.values())
                                    if word_freq else 0)

        features.append(f)

    return pd.DataFrame(features).fillna(0).values


# ── Load & split muril1.npy / muril1.xlsx by language ────────────────────────
def load_all_data():
    npy_path  = HERE / "muril1.npy"
    xlsx_path = HERE / "muril1.xlsx"

    if not npy_path.exists():
        raise FileNotFoundError(f"Missing: {npy_path}")
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Missing: {xlsx_path}")

    log.info("Loading muril1.npy …")
    X_all = np.load(str(npy_path))

    log.info("Loading muril1.xlsx …")
    df = pd.read_excel(str(xlsx_path), engine="openpyxl")

    df.columns = (
        df.columns
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii")
          .str.strip()
          .str.lower()
    )
    log.info(f"Columns found: {list(df.columns)}")

    for col in ["language", "label"]:
        if col not in df.columns:
            raise ValueError(
                f"Required column '{col}' not found in muril1.xlsx")

    df = df.dropna(subset=["label"]).reset_index(drop=True)

    if X_all.shape[0] != len(df):
        raise ValueError(
            f"muril1.npy has {X_all.shape[0]} rows but "
            f"muril1.xlsx has {len(df)} rows."
        )

    df["language"] = df["language"].astype(str).str.strip().str.lower()
    df["label"]    = df["label"].astype(str).str.strip().str.lower()
    label_map      = {v: (1 if v in ("fake", "1", "yes", "true") else 0)
                      for v in df["label"].unique()}
    df["label"]    = df["label"].map(label_map).astype(int)

    log.info(f"Total rows: {len(df)}  "
             f"Languages: {df['language'].value_counts().to_dict()}")

    lang_data = {}
    for lang_raw, model_name in LANGUAGE_MAP.items():
        mask = df["language"] == lang_raw
        if mask.sum() == 0:
            log.warning(
                f"No rows found for language='{lang_raw}' — skipping.")
            continue

        indices = df.index[mask].tolist()
        X_muril = X_all[indices]
        y_lang  = df.loc[mask, "label"].values
        texts   = df.loc[mask, "text"].fillna("").tolist()

        # ── Extract handcrafted features ───────────────────────────────
        log.info(f"  Extracting handcrafted features for {lang_raw}…")
        X_hand = extract_handcrafted_features(texts, lang_raw)

        # ── Combine MuRIL + handcrafted features ──────────────────────
        X_combined = np.hstack([X_muril, X_hand])
        log.info(f"  {model_name}: MuRIL={X_muril.shape[1]}dims + "
                 f"handcrafted={X_hand.shape[1]}dims = "
                 f"{X_combined.shape[1]}dims total")
        log.info(f"  Rows={X_combined.shape[0]}  "
                 f"real={(y_lang==0).sum()}  "
                 f"fake={(y_lang==1).sum()}")

        lang_data[model_name] = (X_combined, y_lang, lang_raw)

    return lang_data


# ── Class imbalance check ─────────────────────────────────────────────────────
def check_imbalance(X_train, y_train, lang_name):
    real_count = (y_train == 0).sum()
    fake_count = (y_train == 1).sum()
    ratio      = min(real_count, fake_count) / max(real_count, fake_count)

    print(f"\n  Class balance:")
    print(f"  Real={real_count}  Fake={fake_count}  Ratio={ratio:.2f}")

    if ratio < 0.8:
        print(f"  ⚠ Imbalanced — applying SMOTE…")
        try:
            from imblearn.over_sampling import SMOTE
            sm              = SMOTE(random_state=RANDOM_STATE)
            X_train, y_train = sm.fit_resample(X_train, y_train)
            print(f"  After SMOTE: "
                  f"Real={(y_train==0).sum()}  "
                  f"Fake={(y_train==1).sum()}")
        except ImportError:
            print("  imbalanced-learn not installed — skipping SMOTE")
            print("  Run: pip install imbalanced-learn")
    else:
        print(f"  ✓ Balanced — no SMOTE needed")

    return X_train, y_train


# ── CV evaluation on TRAIN set only ──────────────────────────────────────────
def evaluate(X_train, y_train) -> dict:
    skf     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)
    results = {}
    for clf_name, clf in CLASSIFIERS.items():
        pipe    = Pipeline([("scaler", StandardScaler()),
                            ("clf",    clf)])
        t0      = time.time()
        scores  = cross_val_score(pipe, X_train, y_train,
                                  cv=skf, scoring="accuracy",
                                  n_jobs=-1)
        elapsed = time.time() - t0
        results[clf_name] = {
            "mean_acc": float(scores.mean()),
            "std_acc":  float(scores.std()),
            "scores":   scores.tolist(),
            "time_sec": round(elapsed, 2),
        }
        print(f"    {clf_name:<22}  "
              f"acc={scores.mean():.4f} ± {scores.std():.4f}  "
              f"[{elapsed:.1f}s]")
    return results


# ── Hyperparameter tuning ─────────────────────────────────────────────────────
def tune_classifier(X_train, y_train, clf_name):
    if clf_name not in PARAM_GRIDS:
        log.info(f"  No param grid for {clf_name} — skipping tuning")
        return None

    print(f"\n  Tuning hyperparameters for {clf_name}…")
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("clf",    CLASSIFIERS[clf_name])])
    grid = GridSearchCV(pipe, PARAM_GRIDS[clf_name],
                        cv=3, scoring="accuracy",
                        n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)
    print(f"  Best params : {grid.best_params_}")
    print(f"  Best CV acc : {grid.best_score_:.4f}")
    return grid.best_estimator_


# ── Train best classifier ─────────────────────────────────────────────────────
def train_best(X_train, y_train, clf_name) -> Pipeline:
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("clf",    CLASSIFIERS[clf_name])])
    pipe.fit(X_train, y_train)
    return pipe


# ── ROC-AUC evaluation ────────────────────────────────────────────────────────
def evaluate_roc(model, X_test, y_test):
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_proba = model.decision_function(X_test)
    else:
        return None

    auc = roc_auc_score(y_test, y_proba)
    print(f"\n  ROC-AUC Score : {auc:.4f}")
    print(f"  (1.0=perfect  0.5=random guessing)")
    return float(auc)


# ── Learning curve ────────────────────────────────────────────────────────────
def compute_learning_curve(X_train, y_train, clf_name, lang_name):
    print(f"\n  Computing learning curve for {clf_name}…")
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("clf",    CLASSIFIERS[clf_name])])

    train_sizes, train_scores, val_scores = learning_curve(
        pipe, X_train, y_train,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring="accuracy", n_jobs=-1
    )

    curve_data = {
        "train_sizes":  train_sizes.tolist(),
        "train_scores": train_scores.mean(axis=1).tolist(),
        "val_scores":   val_scores.mean(axis=1).tolist(),
        "train_std":    train_scores.std(axis=1).tolist(),
        "val_std":      val_scores.std(axis=1).tolist(),
    }

    curve_path = MODEL_DIR / f"{lang_name}_learning_curve.json"
    with open(str(curve_path), "w") as f:
        json.dump(curve_data, f, indent=2)

    print(f"\n  {'Train Size':>12} {'Train Acc':>12} {'Val Acc':>12}")
    print(f"  {'-'*38}")
    for sz, tr, vl in zip(
            train_sizes,
            curve_data["train_scores"],
            curve_data["val_scores"]):
        gap    = tr - vl
        status = "overfit" if gap > 0.02 else "good"
        print(f"  {int(sz):>12} {tr:>12.4f} {vl:>12.4f}  [{status}]")

    print(f"\n  Saved → {curve_path}")
    return curve_data


# ── Confusion matrix ──────────────────────────────────────────────────────────
def print_confusion_matrix(cm, lang_name):
    tn, fp, fn, tp = cm.ravel()
    print(f"\n  Confusion Matrix — {lang_name}")
    print(f"  {'':20} {'Pred REAL':>12} {'Pred FAKE':>12}")
    print(f"  {'Actual REAL':20} {tn:>12} {fp:>12}")
    print(f"  {'Actual FAKE':20} {fn:>12} {tp:>12}")
    print(f"\n  TP (correctly caught fake) : {tp}")
    print(f"  TN (correctly passed real) : {tn}")
    print(f"  FP (real marked as fake)   : {fp}  ← Type I error")
    print(f"  FN (fake passed as real)   : {fn}  ← Type II error")
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = (2 * precision * recall / max(precision + recall, 1e-9))
    print(f"\n  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")


# ── Main ──────────────────────────────────────────────────────────────────────
def run():
    lang_data    = load_all_data()
    full_report  = {}
    best_overall = {"embedding": None, "classifier": None, "accuracy": 0.0}
    lang_config  = {}

    for lang_name, (X, y, lang_raw) in lang_data.items():
        print(f"\n{'='*60}")
        print(f"  Language : {lang_name}  ({lang_raw})")
        print(f"  Samples  : {len(y)}  "
              f"real={(y==0).sum()}  fake={(y==1).sum()}")
        print(f"  Features : {X.shape[1]} dims "
              f"(768 MuRIL + handcrafted)")
        print(f"{'='*60}")

        # ── 80/20 train/test split ─────────────────────────────────────
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y
        )
        print(f"\n  Train : {len(y_train)} samples")
        print(f"  Test  : {len(y_test)} samples (held out)")

        # ── Class imbalance check ──────────────────────────────────────
        X_train, y_train = check_imbalance(X_train, y_train, lang_name)

        # ── Cross-validation on train set ──────────────────────────────
        print(f"\n  Running {CV_FOLDS}-fold CV on "
              f"{len(CLASSIFIERS)} classifiers…")
        results  = evaluate(X_train, y_train)
        best_clf = max(results, key=lambda n: results[n]["mean_acc"])
        best_acc = results[best_clf]["mean_acc"]
        print(f"\n  ★  Best: {best_clf}  (CV acc={best_acc:.4f})")

        # ── Hyperparameter tuning ──────────────────────────────────────
        tuned = tune_classifier(X_train, y_train, best_clf)
        if tuned is not None:
            model = tuned
            print(f"  Using tuned model")
        else:
            model = train_best(X_train, y_train, best_clf)
            print(f"  Using default model")

        # ── Learning curve ─────────────────────────────────────────────
        curve_data = compute_learning_curve(
            X_train, y_train, best_clf, lang_name)

        # ── Evaluate on unseen test set ────────────────────────────────
        print(f"\n  ── Honest evaluation on UNSEEN test set ──")
        y_pred   = model.predict(X_test)
        test_acc = float((y_pred == y_test).mean())
        cm       = confusion_matrix(y_test, y_pred)
        auc      = evaluate_roc(model, X_test, y_test)

        print(f"  Test accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
        print(classification_report(
            y_test, y_pred, target_names=["real", "fake"]))
        print_confusion_matrix(cm, lang_name)

        # ── Save model ─────────────────────────────────────────────────
        out_path = MODEL_DIR / f"{lang_name}_model.pkl"
        with open(str(out_path), "wb") as f:
            pickle.dump({
                "model":            model,
                "embedding_model":  "google/muril-base-cased",
                "lang_name":        lang_name,
                "lang_raw":         lang_raw,
                "classifier_name":  best_clf,
                "cv_accuracy":      best_acc,
                "test_accuracy":    test_acc,
                "roc_auc":          auc,
                "confusion_matrix": cm.tolist(),
                "feature_dims": {
                    "muril":        768,
                    "handcrafted":  X.shape[1] - 768,
                    "total":        X.shape[1],
                },
            }, f)
        print(f"\n  Saved → {out_path}")

        full_report[lang_name] = {
            "best_classifier":  best_clf,
            "best_cv_accuracy": best_acc,
            "test_accuracy":    test_acc,
            "roc_auc":          auc,
            "confusion_matrix": cm.tolist(),
            "learning_curve":   curve_data,
            "all_results":      results,
        }
        lang_config[lang_name] = {
            "classifier":    best_clf,
            "accuracy":      best_acc,
            "test_accuracy": test_acc,
            "roc_auc":       auc,
        }

        if best_acc > best_overall["accuracy"]:
            best_overall = {
                "embedding":  lang_name,
                "classifier": best_clf,
                "accuracy":   best_acc,
            }

    # ── Summary ───────────────────────────────────────────────────────
    print("\n\n" + "=" * 60)
    print("  SUMMARY")
    print("=" * 60)
    print(f"{'Language':<16} {'Classifier':<22} "
          f"{'CV Acc':>8} {'Test Acc':>10} {'AUC':>8}")
    print("-" * 68)
    for lang, info in full_report.items():
        marker = " ◀ BEST" if lang == best_overall["embedding"] else ""
        auc_str = (f"{info['roc_auc']:.4f}"
                   if info["roc_auc"] else "N/A")
        print(f"{lang:<16} {info['best_classifier']:<22} "
              f"{info['best_cv_accuracy']:>8.4f} "
              f"{info['test_accuracy']:>10.4f} "
              f"{auc_str:>8}{marker}")

    # ── Save reports ───────────────────────────────────────────────────
    report_path = MODEL_DIR / "training_report.json"
    with open(str(report_path), "w") as f:
        json.dump(full_report, f, indent=2)
    print(f"\nFull report  → {report_path}")

    best_path = MODEL_DIR / "best_config.json"
    with open(str(best_path), "w") as f:
        json.dump(best_overall, f, indent=2)
    print(f"Best config  → {best_path}")

    lang_path = MODEL_DIR / "lang_config.json"
    with open(str(lang_path), "w") as f:
        json.dump(lang_config, f, indent=2)
    print(f"Lang config  → {lang_path}")

    print(f"\nNow start the server:  python app.py")


if __name__ == "__main__":
    run()


  Language : hindi1  (hindi)
  Samples  : 20593  real=10293  fake=10300
  Features : 784 dims (768 MuRIL + handcrafted)

  Train : 16474 samples
  Test  : 4119 samples (held out)

  Class balance:
  Real=8234  Fake=8240  Ratio=1.00
  ✓ Balanced — no SMOTE needed

  Running 5-fold CV on 10 classifiers…
    LogisticRegression      acc=0.9938 ± 0.0016  [5.4s]
    LinearSVC               acc=0.9943 ± 0.0013  [57.1s]
    SVC_RBF                 acc=0.9956 ± 0.0009  [135.0s]
    SGD                     acc=0.9928 ± 0.0012  [5.7s]
    RandomForest            acc=0.9927 ± 0.0013  [438.1s]
    ExtraTrees              acc=0.9924 ± 0.0018  [55.2s]
    GradientBoosting        acc=0.9913 ± 0.0013  [2018.3s]
    AdaBoost                acc=0.9898 ± 0.0020  [762.3s]
    GaussianNB              acc=0.9299 ± 0.0046  [1.6s]
    MLP                     acc=0.9962 ± 0.0011  [71.3s]

  ★  Best: MLP  (CV acc=0.9962)

  Tuning hyperparameters for MLP…
  Best params : {'clf__alpha': 0.001, 'clf__hidden_layer

In [4]:
"""
app.py — Flask REST API for the Fake News Detector Chrome extension.
Supports Hindi, Marathi, Telugu, Gujarati using muril-base-cased embeddings.
Each language has its own trained model.

Start:  python app.py
Test:   curl http://127.0.0.1:5000/health
"""

from dotenv import load_dotenv
load_dotenv()

import json, pickle, time, logging, warnings, re
from pathlib import Path

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS
import os

# ── Load HF token ─────────────────────────────────────────────────────────────
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    env_file = Path(__file__).parent / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith("HF_TOKEN="):
                hf_token = line.split("=", 1)[1].strip()
                break
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

# ── Suppress noisy warnings ───────────────────────────────────────────────────
warnings.filterwarnings("ignore", message=".*Some weights of the model checkpoint.*")
warnings.filterwarnings("ignore", message=".*were not used.*")
warnings.filterwarnings("ignore", message=".*initializing.*")
warnings.filterwarnings("ignore", message=".*not sharded.*")
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
logging.getLogger("huggingface_hub.file_download").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────────
HERE      = Path(__file__).parent
MODEL_DIR = HERE / "models"

HF_MODEL_NAME = "google/muril-base-cased"

# ── Language → model name mapping ─────────────────────────────────────────────
LANG_TO_MODEL = {
    "hindi":    "hindi1",
    "marathi":  "marathi1",
    "telugu":   "telugu1",
    "gujarati": "gujarati1",
}

# ── Native language label translations ────────────────────────────────────────
LABEL_TRANSLATIONS = {
    "hindi": {
        "fake": "❌ फ़ेक न्यूज़ (नकली खबर)",
        "real": "✅ असली खबर (सत्य)",
    },
    "marathi": {
        "fake": "❌ बनावट बातमी (खोटी)",
        "real": "✅ खरी बातमी (सत्य)",
    },
    "telugu": {
        "fake": "❌ నకిలీ వార్త (అసత్యం)",
        "real": "✅ నిజమైన వార్త (సత్యం)",
    },
    "gujarati": {
        "fake": "❌ નકલી સમાચાર (ખોટા)",
        "real": "✅ સાચા સમાચાર (સત્ય)",
    },
}

# ── Fake/real signal words per language ───────────────────────────────────────
FAKE_SIGNALS = {
    "hindi":    ["वायरल", "खुलासा", "चौंकाने", "शेयर", "सच्चाई",
                 "breaking", "viral", "exclusive", "shocking",
                 "अलर्ट", "सावधान", "दावा", "झूठ"],
    "marathi":  ["व्हायरल", "खुलासा", "धक्कादायक", "शेअर",
                 "सत्य", "breaking", "viral", "दावा", "खोटे"],
    "telugu":   ["వైరల్", "షాకింగ్", "వెల్లడి", "షేర్",
                 "నిజం", "breaking", "viral", "దావా"],
    "gujarati": ["વાયરલ", "ખુલાસો", "ચોંકાવนारું", "શेर",
                 "સત્ય", "breaking", "viral", "દાવો", "ખોटા"],
}

REAL_SIGNALS = {
    "hindi":    ["पीटीआई", "एएनआई", "संसद", "सरकार", "अधिकारी",
                 "बयान", "रिपोर्ट", "PTI", "ANI", "प्रेस"],
    "marathi":  ["पीटीआई", "एएनआई", "संसद", "सरकार", "अधिकारी",
                 "निवेदन", "अहवाल", "PTI", "ANI"],
    "telugu":   ["పిటిఐ", "ప్రభుత్వం", "పార్లమెంట్", "అధికారి",
                 "నివేదిక", "PTI", "ANI", "ప్రకటన"],
    "gujarati": ["પีटीઆઈ", "સরકાર", "સંसद", "અधिकारी",
                 "નિવેદन", "અહेવाল", "PTI", "ANI"],
}

# ── langdetect code → normalized language name ────────────────────────────────
LANG_CODES = {
    "hi": "hindi",
    "mr": "marathi",
    "te": "telugu",
    "gu": "gujarati",
}

# ── Language detection ─────────────────────────────────────────────────────────
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 42
    USE_LANGDETECT = True
except ImportError:
    log.info("langdetect not found — attempting to install…")
    import subprocess, sys
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "langdetect", "-q"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 42
        USE_LANGDETECT = True
        log.info("langdetect installed successfully.")
    except Exception as install_err:
        USE_LANGDETECT = False
        log.warning(f"langdetect could not be installed: {install_err}")

# ── Flask app ──────────────────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

# ── Globals ────────────────────────────────────────────────────────────────────
_classifiers = {}
_embed_fn    = None
_lang_config = {}


# ── Text preprocessing ─────────────────────────────────────────────────────────
def preprocess_text(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Handcrafted features (must match train.py exactly) ────────────────────────
def extract_handcrafted_features(text: str, lang: str) -> np.ndarray:
    fake_words = FAKE_SIGNALS.get(lang, FAKE_SIGNALS["hindi"])
    real_words = REAL_SIGNALS.get(lang, REAL_SIGNALS["hindi"])

    words      = text.split()
    sentences  = re.split(r"[।.!?]", text)
    sentences  = [s.strip() for s in sentences if s.strip()]
    text_lower = text.lower()

    word_freq = {}
    for w in words:
        word_freq[w] = word_freq.get(w, 0) + 1

    fake_count = sum(1 for w in fake_words if w.lower() in text_lower)
    real_count = sum(1 for w in real_words if w.lower() in text_lower)

    features = [
        # Lexical
        len(words),
        len(text),
        sum(len(w) for w in words) / max(len(words), 1),
        len(set(words)) / max(len(words), 1),
        # Punctuation
        text.count("!"),
        text.count("?"),
        text.count("..."),
        sum(1 for c in text if c.isupper()) / max(len(text), 1),
        # Signal words
        fake_count,
        real_count,
        fake_count / max(fake_count + real_count, 1),
        # Structural
        len(sentences),
        sum(len(s.split()) for s in sentences) / max(len(sentences), 1),
        # URL and numbers
        len(re.findall(r"http\S+|www\.\S+", text)),
        len(re.findall(r"\d+", text)),
        # Repetition
        max(word_freq.values()) if word_freq else 0,
    ]

    return np.array(features, dtype=float).reshape(1, -1)


# ── Calibrate overconfident probabilities ─────────────────────────────────────
def calibrate_confidence(raw_conf: float) -> float:
    if raw_conf >= 0.95:
        calibrated = 0.87 + (raw_conf - 0.95) * (0.08 / 0.05)
        return round(min(calibrated, 0.95), 4)
    return round(raw_conf, 4)


# ── Resource loading ───────────────────────────────────────────────────────────
def load_resources():
    global _classifiers, _embed_fn, _lang_config

    lang_cfg_path = MODEL_DIR / "lang_config.json"
    if not lang_cfg_path.exists():
        raise FileNotFoundError(
            "models/lang_config.json not found. Run train.py first.")
    with open(str(lang_cfg_path)) as f:
        _lang_config = json.load(f)
    log.info(f"Loaded lang_config: {list(_lang_config.keys())}")

    for lang_name in LANG_TO_MODEL.values():
        pkl_path = MODEL_DIR / f"{lang_name}_model.pkl"
        if not pkl_path.exists():
            log.warning(f"Model not found: {pkl_path} — skipping.")
            continue
        with open(str(pkl_path), "rb") as f:
            data = pickle.load(f)
        _classifiers[lang_name] = data["model"]
        log.info(f"Loaded [{lang_name}] → "
                 f"{data.get('classifier_name','?')}  "
                 f"cv={data.get('cv_accuracy',0):.4f}  "
                 f"test={data.get('test_accuracy','N/A')}  "
                 f"auc={data.get('roc_auc','N/A')}")

    if not _classifiers:
        raise RuntimeError("No models loaded. Run train.py first.")

    log.info(f"Loading embedding model: {HF_MODEL_NAME} …")
    _embed_fn = _build_embed_fn(HF_MODEL_NAME)
    log.info("Embedding model ready.")


def _build_embed_fn(hf_name: str):
    try:
        from sentence_transformers import SentenceTransformer
        st_model = SentenceTransformer(hf_name)
        log.info(f"Using sentence-transformers for {hf_name}")
        def embed_st(text: str) -> np.ndarray:
            return st_model.encode([text])
        return embed_st
    except Exception as e:
        log.warning(f"sentence-transformers failed ({e}), "
                    "trying HF transformers…")

    try:
        import torch
        from transformers import AutoTokenizer, AutoModel
        tokenizer = AutoTokenizer.from_pretrained(hf_name)
        hf_model  = AutoModel.from_pretrained(
            hf_name, ignore_mismatched_sizes=True)
        hf_model.eval()
        log.info(f"Using HuggingFace transformers for {hf_name}")

        def embed_hf(text: str) -> np.ndarray:
            inputs = tokenizer(text, return_tensors="pt",
                               truncation=True, max_length=512,
                               padding=True)
            with torch.no_grad():
                outputs = hf_model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).numpy()
        return embed_hf
    except Exception as e:
        raise RuntimeError(
            f"Could not load MuRIL. "
            f"Install sentence-transformers or transformers+torch. "
            f"Error: {e}"
        )


def detect_language(text: str):
    if not USE_LANGDETECT:
        return None
    try:
        code = detect(text)
        return LANG_CODES.get(code, None)
    except Exception:
        return None


def resolve_language(requested_lang, text):
    if requested_lang:
        normalized = requested_lang.strip().lower()
        if normalized in LANG_TO_MODEL:
            return normalized
        for key in LANG_TO_MODEL:
            if key in normalized:
                return key
    return detect_language(text)


def get_embedding_and_features(text: str, lang: str) -> np.ndarray:
    """Combine MuRIL embedding + handcrafted features — must match train.py"""
    muril_vec = _embed_fn(text)                              # (1, 768)
    hand_vec  = extract_handcrafted_features(text, lang)     # (1, 16)
    return np.hstack([muril_vec, hand_vec])                  # (1, 784)


# ── Routes ─────────────────────────────────────────────────────────────────────

@app.route("/favicon.ico")
def favicon():
    return ("", 204)


@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "service":             "Fake News Detector API",
        "version":             "3.0",
        "status":              "running",
        "supported_languages": list(LANG_TO_MODEL.keys()),
        "loaded_models":       list(_classifiers.keys()),
        "embedding_model":     HF_MODEL_NAME,
        "feature_engineering": "MuRIL 768-dim + 16 handcrafted = 784 total",
        "endpoints": {
            "GET  /health":        "Model status and metadata",
            "POST /predict":       "Single text prediction",
            "POST /predict/batch": "Batch prediction (max 50 items)",
        },
        "example": {
            "method": "POST /predict",
            "body": {
                "text":     "Your news article text here…",
                "language": "hindi",
            },
        },
    })


@app.route("/health", methods=["GET"])
def health():
    model_status = {}
    for lang, model_name in LANG_TO_MODEL.items():
        cfg = _lang_config.get(model_name, {})
        model_status[lang] = {
            "model":        model_name,
            "loaded":       model_name in _classifiers,
            "classifier":   cfg.get("classifier", "unknown"),
            "cv_accuracy":  cfg.get("accuracy", None),
            "test_accuracy": cfg.get("test_accuracy", None),
            "roc_auc":      cfg.get("roc_auc", None),
        }
    return jsonify({
        "status":          "ok",
        "embedding_model": HF_MODEL_NAME,
        "languages":       model_status,
    })


@app.route("/predict", methods=["POST"])
def predict():
    t0   = time.time()
    body = request.get_json(silent=True) or {}

    text = preprocess_text((body.get("text") or "").strip())
    if len(text) < 10:
        return jsonify({"error": "text too short (min 10 chars)"}), 400

    if not _classifiers or _embed_fn is None:
        return jsonify({"error": "models not loaded"}), 503

    lang = resolve_language(body.get("language"), text)
    if lang is None:
        return jsonify({
            "error": "Could not detect language. "
                     f"Please provide one of: {list(LANG_TO_MODEL.keys())}"
        }), 400

    model_name = LANG_TO_MODEL.get(lang)
    if model_name not in _classifiers:
        return jsonify({
            "error": f"Model for '{lang}' not loaded. "
                     f"Available: {list(_classifiers.keys())}"
        }), 400

    try:
        X = get_embedding_and_features(text, lang)
    except Exception as e:
        log.error(f"Feature extraction failed: {e}")
        return jsonify({"error": f"feature extraction error: {e}"}), 500

    classifier = _classifiers[model_name]
    pred       = int(classifier.predict(X)[0])
    label      = "fake" if pred == 1 else "real"

    confidence = None
    if hasattr(classifier, "predict_proba"):
        proba      = classifier.predict_proba(X)[0]
        confidence = calibrate_confidence(float(max(proba)))
    elif hasattr(classifier, "decision_function"):
        score      = classifier.decision_function(X)[0]
        confidence = calibrate_confidence(
            float(1 / (1 + np.exp(-abs(score)))))

    translations = LABEL_TRANSLATIONS.get(lang, {})
    label_native = translations.get(label, label)
    cfg          = _lang_config.get(model_name, {})
    elapsed      = int((time.time() - t0) * 1000)

    log.info(f"Predict → lang={lang}  label={label}  "
             f"conf={confidence}  time={elapsed}ms")

    return jsonify({
        "label":        label,
        "label_native": label_native,
        "confidence":   confidence,
        "language":     lang,
        "model":        model_name,
        "classifier":   cfg.get("classifier", "unknown"),
        "embedding":    HF_MODEL_NAME,
        "time_ms":      elapsed,
    })


@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    body  = request.get_json(silent=True) or {}
    items = body.get("items", [])
    if not items:
        return jsonify({"error": "no items provided"}), 400

    results = []
    for item in items[:50]:
        text = preprocess_text((item.get("text") or "").strip())
        if len(text) < 10:
            results.append({"error": "text too short"})
            continue

        lang = resolve_language(item.get("language"), text)
        if lang is None:
            results.append({"error": "could not detect language"})
            continue

        model_name = LANG_TO_MODEL.get(lang)
        if model_name not in _classifiers:
            results.append({"error": f"model for '{lang}' not loaded"})
            continue

        try:
            X          = get_embedding_and_features(text, lang)
            classifier = _classifiers[model_name]
            pred       = int(classifier.predict(X)[0])
            label      = "fake" if pred == 1 else "real"
            conf       = None
            if hasattr(classifier, "predict_proba"):
                conf = calibrate_confidence(
                    float(max(classifier.predict_proba(X)[0])))
            elif hasattr(classifier, "decision_function"):
                score = classifier.decision_function(X)[0]
                conf  = calibrate_confidence(
                    float(1 / (1 + np.exp(-abs(score)))))

            translations = LABEL_TRANSLATIONS.get(lang, {})
            label_native = translations.get(label, label)
            cfg          = _lang_config.get(model_name, {})

            results.append({
                "label":        label,
                "label_native": label_native,
                "confidence":   conf,
                "language":     lang,
                "model":        model_name,
                "classifier":   cfg.get("classifier", "unknown"),
                "embedding":    HF_MODEL_NAME,
            })
        except Exception as e:
            results.append({"error": str(e)})

    return jsonify({"results": results})


# ── Entry ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    load_resources()
    app.run(host="127.0.0.1", port=5000, debug=False)

NameError: name '__file__' is not defined